# Quantum Workflow Example

This notebook provides a tutorial of how to use the tools to generate quantum enhanced features outlined by [1]. The tools developed in the src directory and used in this notebook are easily extendible to different feature maps, circuit execution modes, and observables. As an example, this notebook generates quantum features with a standard unitary encoding circuit, and pauli correlation encoding to map observables to output features.

## References

[1] Rigetti Computing. (2023, April 25). *Recession prediction via signature kernels enhanced with quantum features*. Medium. https://medium.com/rigetti/recession-prediction-via-signature-kernels-enhanced-with-quantum-features-a608995d48f7

In [31]:
# imports from developed code
from src.quantum_utils.engineer_features import engineer_features
from src.quantum_utils.encoding_strategy.su_encoding_strategy import su_encoding_strategy
from src.quantum_utils.result_getters.pce_result_getter import pce_result_getter

In [32]:
# import Dataset
import pandas as pd

data_df = pd.read_parquet("data/processed/track_B_curated.parquet")
data_df.dropna(inplace=True)

## Core Workflow

The relationships that are captured with this method are affected by: 
- A) How classical input data is encoded into a quantum state
- B) How a quantum state is encoded in classical data

Although both stages are sometimes referred to as "encoding", however they serve fundamentally different purposes and are not generally interchangeable. Using identical mappings for both stages produces a trivial transformation with limited representational benefit.

The purpose of this framework is to provide a modular architecture where encoding strategies and result extraction methods can be independently developed, reused, and combined during experimentation.

---

## Workflow Overview

The feature engineering pipeline consists of four main steps:

### 1. Define the Qiskit Primitive and Number of Qubits

The Qiskit primitive (such as an estimator or sampler) and the number of qubits are initialized outside the framework classes. This simplifies experimentation within a runtime session and avoids repeatedly re-instantiating backend objects.

The number of qubits is defined first because it is required by both the encoding strategy and the result getter.

---

### 2. Define an Encoding Strategy

The encoding strategy specifies:

- The parameterized quantum circuit used to prepare the quantum state
- How classical input features are transformed into circuit parameter values

This component determines how information from the classical dataset is embedded into the Hilbert space representation.

---

### 3. Define a Result Getter

The result getter defines how the encoded quantum state is evaluated and translated back into classical information.

Depending on the implementation, this may involve:

- Simulation and/or hardware settings
- The set of Observables that will be measured
- post-processing on said observables to obtain expectation values

The output of this stage is a set of engineered classical features derived from the quantum representation.

---

### 4. Run `engineer_features`

The `engineer_features()` function combines:

- The encoding strategy
- The result getter
- The input dataset

and produces a transformed dataset containing quantum-enhanced features suitable for downstream machine learning tasks.

In [33]:
# imports from developed code
from src.quantum_utils.engineer_features import engineer_features
from src.quantum_utils.encoding_strategy.su_encoding_strategy import su_encoding_strategy
from src.quantum_utils.result_getters.pce_result_getter import pce_result_getter
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Estimator

sim = AerSimulator()
estimator = Estimator(mode=sim)

num_qubits = 3

# Define preprocessing and feature map settings.
encoding_strategy = su_encoding_strategy(n_qubits=num_qubits, 
                                         reps=1, 
                                         su2_gates=None, 
                                         entanglement='full')

# Define settings to execute circuit and extract features.
result_getter = pce_result_getter(num_qubits=num_qubits,
                                  num_features=9,
                                  num_operators=2,
                                  #estimator=estimator,
                                )

quantum_features = engineer_features(data_df, 
                  encoding_strategy, 
                  result_getter)

print("Dataframe of engineered quantum features:")
quantum_features.head()

Dataframe of engineered quantum features:


,XXI,XIX,IXX,YYI,YIY,IYY,ZZI,ZIZ,IZZ
0,0.097051,-0.057513,-0.592606,0.045276,-1.534562e-02,-0.589606,-0.466523,0.464162,-0.994939
1,0.137648,-0.122489,-0.889872,0.020851,6.938894e-18,-0.880788,-0.151480,0.149933,-0.989792
2,0.256679,-0.222291,-0.866025,0.010208,0.000000e+00,-0.836008,-0.039770,0.038391,-0.965339
3,0.288005,-0.210546,-0.731048,0.002171,-1.025920e-03,-0.698101,-0.007539,0.007199,-0.954932
4,0.354950,-0.307396,-0.866025,0.111823,0.000000e+00,-0.804797,-0.315040,0.292766,-0.929299


In [34]:
enhanced_df = pd.concat([data_df.reset_index(drop=True), quantum_features.reset_index(drop=True)], axis=1)
enhanced_df.head()

,UNRATE,PERMIT,S&P 500,UMCSENTx,T10Y3M_level,T10Y3M_delta,XXI,XIX,IXX,YYI,YIY,IYY,ZZI,ZIZ,IZZ
0,-0.1,7.040536,-0.077267,-4.5,1.18,0.07,0.097051,-0.057513,-0.592606,0.045276,-1.534562e-02,-0.589606,-0.466523,0.464162,-0.994939
1,0.0,7.050989,-0.124253,0.0,1.18,0.00,0.137648,-0.122489,-0.889872,0.020851,6.938894e-18,-0.880788,-0.151480,0.149933,-0.989792
2,-0.1,7.080868,0.023802,0.0,1.09,-0.09,0.256679,-0.222291,-0.866025,0.010208,0.000000e+00,-0.836008,-0.039770,0.038391,-0.965339
3,0.3,7.090077,0.026844,-3.8,1.16,0.07,0.288005,-0.210546,-0.731048,0.002171,-1.025920e-03,-0.698101,-0.007539,0.007199,-0.954932
4,-0.1,7.109062,-0.008926,0.0,1.20,0.04,0.354950,-0.307396,-0.866025,0.111823,0.000000e+00,-0.804797,-0.315040,0.292766,-0.929299


The enhanced_df can then be used in any learning pipeline that would have been applied to data_df. In particular, using this method with a signature kernel + support vector machine pipeline has been shown to increate prediciton accuracy.